# Category selection (Issue #14)

Step 1: **Analyze category support** count positive/negative examples per category and identify
categories with too few/many examples for reliable training/evaluation.

Reuses the loading code from `01_load_cuad.ipynb` via `%run` (temporary — will move
shared functions into a `.py` module later).

In [8]:
%run 01_load_cuad.ipynb

train: 408 contracts, 11180 clause spans
test:  102 contracts, 2643 clause spans
train categories: 41
test categories:  40
in train but not test: {'Price Restrictions'}
train offset match rate: 1.0
test offset match rate:  1.0
(408, 41) (102, 40)


The cell above just executed the whole `01_load_cuad.ipynb`[Jay's Issue] notebook (print output). We now have, in this notebook's namespace:

- `train_labels`, `test_labels` — contract x category multi-hot matrices (already built by 01)
- `train_clauses`, `train_contracts`, `test_clauses`, `test_contracts` — the flattened data
- `load_cuad()`, `label_matrix()`, `category_from_id()` — the loading functions, if we need to reload anything

We'll work from `train_labels` only (not test), so this label-selection decision doesn't peek
at test data.

## Category support

For each of the 41 categories: how many contracts have at least one span of that type,
and what percent of the 408 training contracts that is.

In [9]:
n_contracts = len(train_contracts)

support = train_labels.sum().rename("n_contracts").to_frame()
support["pct_contracts"] = (support["n_contracts"] / n_contracts * 100).round(1)
support = support.sort_values("n_contracts", ascending=False)

support

,n_contracts,pct_contracts
category,,
Document Name,408,100.0
Parties,407,99.8
Agreement Date,377,92.4
Governing Law,354,86.8
Expiration Date,335,82.1
Effective Date,320,78.4
Anti-Assignment,302,74.0
Cap On Liability,231,56.6
License Grant,205,50.2


## Flag sparse categories

`MIN_SUPPORT` is a starting threshold, not a final answer. raise/lower it and see how
the sparse list changes before recommending anything to the team (step 4 of the issue).

In [11]:
MIN_SUPPORT = 20  # contracts; revisit this threshold before finalizing

support["sparse"] = support["n_contracts"] < MIN_SUPPORT

print(f"{support['sparse'].sum()} of {len(support)} categories fall below MIN_SUPPORT={MIN_SUPPORT}:")
support[support["sparse"]]

4 of 41 categories fall below MIN_SUPPORT=20:


,n_contracts,pct_contracts,sparse
category,,,
Affiliate License-Licensor,17,4.2,True
Price Restrictions,15,3.7,True
Unlimited/All-You-Can-Eat-License,14,3.4,True
Source Code Escrow,12,2.9,True


## Negative support check

Checking positive count alone misses the opposite failure mode: a category present in
almost every contract has almost no *negative* examples, so precision/recall on "this
clause is absent" is just as unreliable to estimate. A 2026 CUAD audit ([Tekin,
*Abstention Errors and Segment Support in CUAD*](https://arxiv.org/abs/2609.13274)) makes
this point directly: using a combined threshold (>= N positive AND >= N negative
contracts), only 9 of 41 categories pass at N=30, and 18 pass at N=20.

We reuse `MIN_SUPPORT` and flag on *either* side being too thin.

In [12]:
support["n_neg"] = n_contracts - support["n_contracts"]
support["sparse_neg"] = support["n_neg"] < MIN_SUPPORT
support["sparse_either"] = support["sparse"] | support["sparse_neg"]

print(f"{support['sparse_either'].sum()} of {len(support)} categories are thin on positives, negatives, or both:")
support[support["sparse_either"]]

6 of 41 categories are thin on positives, negatives, or both:


,n_contracts,pct_contracts,sparse,n_neg,sparse_neg,sparse_either
category,,,,,,
Document Name,408,100.0,False,0,True,True
Parties,407,99.8,False,1,True,True
Affiliate License-Licensor,17,4.2,True,391,False,True
Price Restrictions,15,3.7,True,393,False,True
Unlimited/All-You-Can-Eat-License,14,3.4,True,394,False,True
Source Code Escrow,12,2.9,True,396,False,True


## Manual read: any merge candidates?

High co-occurrence (see `01_load_cuad.ipynb` / before the EDA) and CUAD's own "Group"
column in `category_descriptions.csv` cluster some categories together (e.g. the date
fields, the license-grant family, Uncapped/Cap on Liability). Co-occurring isn't the same
as redundant, though, so before proposing any merge we read actual annotated spans for the
4 rarest categories to check whether they're conceptually distinct from each other and
from other categories, or just noisy/overlapping restatements.

In [13]:
RAREST_CATS = ["Affiliate License-Licensor", "Price Restrictions",
               "Unlimited/All-You-Can-Eat-License", "Source Code Escrow"]

for cat in RAREST_CATS:
    print(f"=== {cat} ===")
    examples = train_clauses.loc[train_clauses["category"] == cat, "clause_text"].head(3)
    for ex in examples:
        print(" -", ex[:200].replace("\n", " "))
    print()

=== Affiliate License-Licensor ===
 - Subject to the restrictions set out in Section 2.2, ENERGOUS hereby grants to DIALOG a non-exclusive (subject to Section 2.5), irrevocable, worldwide, sub-licensable (solely in accordance with Section
 - "Product IP" means (a) all Intellectual Property Rights in and to the Products, including all Product Updates, (b) any other Inventions and work products created or developed in connection with resear
 - "Customer Property" means all Intellectual Property, together with all materials, data, writings and other property in any form whatsoever, which is (a) owned or controlled by Customer or its Affiliat

=== Price Restrictions ===
 - The Company  also  reserves  the right to                            increase  or  decrease  the price  per unit  based on                            Company   wide   changes   in  unit   prices  to  
 - The prices set forth in                            Section   2.4(a)  shall  be  subject  to   adjustment              

**Finding: Nothing we could merge.** Reading the spans, all four rarest categories are
distinct: affiliate IP grants (Licensor), pricing-adjustment mechanics,
unlimited-usage license terms, and source-code escrow arrangements, and none overlap in
wording with each other or with a more common category. The same holds when reading the
CSV descriptions for the other official "Group" clusters (e.g. Uncapped Liability and Cap
on Liability are complementary, not duplicate; Affiliate License-Licensor vs -Licensee
differ by *which party's* affiliates are covered). Co occurrence in `category_descriptions.csv`
reflects clauses that tend to appear in the same contracts, not conceptual redundancy, so
I **wouldn't** recommending any merges from this EDA alone.

**Working conclusion for step 1 (pending input on legal value):**
- Rarest categories (regardless of exact `MIN_SUPPORT` cutoff): `Affiliate License-Licensor`,
  `Price Restrictions`, `Unlimited/All-You-Can-Eat-License`, `Source Code Escrow`.
- No merges identified.

## Apply the drop decision

Only the 4 rarest categories are drop candidates. `Document Name` and `Parties` are also
flagged above, but for the opposite reason (too common to measure a reliable false positive
rate)

`PENDING_CA_APPROVAL = True` until the CA answers the question about legal value of
the 4 rare categories. flip it to `False` once we have that answer and actually want the
reduced label set.

In [ ]:
DROP_CANDIDATES = ["Affiliate License-Licensor", "Price Restrictions",
                    "Unlimited/All-You-Can-Eat-License", "Source Code Escrow"]
PENDING_CA_APPROVAL = True  # flip to False once the CA confirms these are safe to drop

if PENDING_CA_APPROVAL:
    print("Not dropping yet \u2014 waiting on CA input. Candidates:", DROP_CANDIDATES)
    final_categories = list(train_labels.columns)
else:
    final_categories = [c for c in train_labels.columns if c not in DROP_CANDIDATES]
    print(f"Dropped {len(DROP_CANDIDATES)} categories -> {len(final_categories)} remain")

final_train_labels = train_labels[final_categories]
final_train_labels.shape